# P&ID Medallion Pipeline — Concepts Walkthrough (Bronze → Silver)

This notebook illustrates, end to end, what we have built so far: a **Bronze**
(raw, immutable ingestion) → **Silver** (parse + topology reconstruction) pipeline
for P&ID interoperability exports (DEXPI/Proteus and INGR ISO-15926 PostProc),
on local Spark + Delta Lake.

It demonstrates the key concepts:

- **Bronze** stores the source XML *as-is* — content hash, format detection, project
  code and drawing revision captured, dedup on exact bytes.
- **Silver** *re-houses* the validated `pidtool`/`bppidsys` reconstruction (the
  "crown jewel") — recovering inline valves the raw graph lacks — and emits typed
  tables: components, segments, connections, equipment.
- The **oracle firewall**: the source turnover assignment is carried as *quarantined*
  lineage, never computed on.
- The **`flow_sense`** four-state directional overlay and the **`derived`** provenance
  flag on every reified connection.
- **Format parity**: DEXPI and PostProc flow through one code path into one schema.
- A real-data finding: **`seg_tag` is not unique** (the CDC anchor-collision risk).
- **Stage D**: a declarative expectation suite writes the **`silver_quality`**
  punch list; only two structural invariants hard-fail (fail for bugs, not data).

> **Run order matters.** After any kernel restart, run the cells top-to-bottom.
> Cell 1 *must* be first — it forces the venv's Spark 3.5.1 and blocks the system
> Spark 4 at `/opt/spark`.

## 0. Environment & pinned session

Two things bite on local WSL and are handled here:

1. **Which Spark.** A system `SPARK_HOME=/opt/spark` (Spark 4) shadows the venv's
   Spark 3.5.1 and breaks Delta (`DeltaCatalog` not found). Cell 1 strips it
   *before* `pyspark` is ever imported.
2. **One catalog, one warehouse.** The Hive metastore (`metastore_db/`) holds
   *names → locations*; the warehouse (`spark-warehouse/`) holds the *data*. We
   **pin both** to fixed paths so every session sees the same tables (embedded
   Derby is single-session — don't also run a `!python -m ...` subprocess while
   this notebook's session is live).

In [ ]:
# --- CELL 1 — must run FIRST (before any `import pyspark`) ---
import os, sys
os.environ.pop("SPARK_HOME", None)                       # ignore system /opt/spark (Spark 4)
os.environ["PYTHONPATH"] = os.pathsep.join(
    p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if "/opt/spark" not in p)
sys.path[:] = [p for p in sys.path if "/opt/spark" not in p]
assert "pyspark" not in sys.modules, "Restart the kernel and run THIS cell first."

from pathlib import Path

# repo_root = the ProjectData repo root (the folder containing bronze/ and silver/)
repo_root = Path.cwd()
while not (repo_root / "bronze").is_dir() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
assert (repo_root / "bronze").is_dir(), f"Open this notebook inside the ProjectData repo (cwd={Path.cwd()})"

# --- the variables for this walkthrough ---
source_sample_dir = repo_root / "sample_data"                 # committed synthetic fixtures
source_dir        = repo_root / "data/exports/projectA"       # real Project A (DEXPI)
source_dir_B      = repo_root / "data/exports/projectB"       # real Project B (PostProc)
bronze_table      = "bronze.pid_documents"                    # NAMED Bronze table (metastore)
spark_warehouse   = repo_root / "spark-warehouse"             # managed-table data
metastore_db      = repo_root / "metastore_db"                # Hive/Derby catalog

# --- metastore hygiene (run BEFORE the session, Cell 2) ---
# Embedded Derby allows ONE connection. A leftover lock from a crashed or still-open
# kernel makes a new session fail with "Unable to instantiate SessionHiveMetaStoreClient".
# Clear stale locks here; flip RESET=True for a guaranteed clean slate — the metastore
# + warehouse are a throwaway PoC store, everything is rebuilt from the XML below.
import shutil
RESET = False        # set True, re-run this cell, then run the notebook top-to-bottom
if RESET:
    shutil.rmtree(metastore_db, ignore_errors=True)
    shutil.rmtree(spark_warehouse, ignore_errors=True)
for _lck in ("db.lck", "dbex.lck"):            # release a stale Derby lock (safe if unheld)
    try:
        (metastore_db / _lck).unlink(missing_ok=True)
    except Exception:
        pass
(repo_root / "derby.log").unlink(missing_ok=True)

print("repo_root       :", repo_root)
print("bronze_table    :", bronze_table)
print("spark_warehouse :", spark_warehouse)
print("metastore_db    :", metastore_db)
print("RESET           :", RESET)

In [ ]:
# --- CELL 2 — build ONE pinned Delta+Hive session (venv Spark 3.5.1) ---
from bronze.spark_session import get_spark
spark = get_spark(extra_conf={
    "spark.sql.warehouse.dir": f"file:{spark_warehouse}",
    "spark.hadoop.javax.jdo.option.ConnectionURL":
        f"jdbc:derby:;databaseName={metastore_db};create=true",
})
import pyspark
from pyspark.sql import functions as F
print("pyspark :", pyspark.__file__)   # expect .../.venv/...  NOT /opt/spark
print("Spark   :", spark.version)      # expect 3.5.1
assert "/opt/spark" not in pyspark.__file__, "Still on system Spark 4 — restart kernel, run Cell 1 first."
assert spark.version.startswith("3.5"), f"Expected Spark 3.5.x, got {spark.version}"
print("OK — venv Spark 3.5.1, Delta + Hive ready.")

In [ ]:
# --- CELL 3 (optional) — targeted rebuild (needs a WORKING metastore) ---
# Drops the named Bronze + Silver tables and their warehouse dirs so a re-run
# starts fresh (avoids DELTA_CREATE_TABLE_WITH_NON_EMPTY_LOCATION). If the
# metastore itself is wedged ("Unable to instantiate SessionHiveMetaStoreClient"),
# this cell can't help — use Cell 1's RESET=True instead (a filesystem wipe that
# runs before the session). Everything is rebuilt from the source XML below.
import shutil
_wipe = {
    "bronze": ["pid_documents"],
    "silver": ["silver_components", "silver_segments", "silver_connections",
               "silver_equipment", "silver_quality"],
}
for db, tables in _wipe.items():
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db}")
    for t in tables:
        spark.sql(f"DROP TABLE IF EXISTS {db}.{t}")
        shutil.rmtree(spark_warehouse / f"{db}.db" / t, ignore_errors=True)
print("clean slate ready")

## 1. Bronze — raw, immutable ingestion

Bronze lands each source file **verbatim**, one row per distinct byte-version, with:
`content` (raw bytes), a self-describing `content_hash` (`sha256:…`), the detected
`source_format` (DEXPI vs POSTPROC), the EPC `document_number` and derived
`project_code`, and the current `drawing_revision` / `drawing_revision_date`.
It **never interprets** the network model — that's Silver's job.

We ingest into the **named** Bronze table `bronze.pid_documents` in the pinned
metastore, and every stage below reads and writes named tables in that one
catalog — so the whole notebook is consistent (no path-based side door).

In [ ]:
# --- pick sources: prefer the real exports, fall back to the committed samples ---
def xmls(d): return sorted(Path(d).glob("*.xml")) if Path(d).is_dir() else []
sources = [d for d in (source_dir, source_dir_B) if xmls(d)]
if not sources:
    sources = [source_sample_dir]
for d in sources:
    print(f"{len(xmls(d)):3d} xml  in  {d}")

In [ ]:
# --- ingest each source folder into the SAME named Bronze table ---
from bronze.notebook import ingest_folder
for d in sources:
    summary = ingest_folder(spark, source_dir=str(d), table_name=bronze_table)
    print(d.name, "->", summary)

In [ ]:
# --- inspect Bronze: both formats, lineage columns, self-describing hash ---
bronze = spark.table(bronze_table)
print("Bronze rows:", bronze.count())
bronze.groupBy("source_format").count().show()
bronze.select("document_number", "drawing_revision", "drawing_revision_date",
              "project_code", "content_hash").show(6, False)

**Dedup on exact bytes.** Re-ingesting the same files lands *nothing* new —
Bronze versions files by `content_hash`, so identical bytes are skipped
(`rows_skipped_already_present`).

In [ ]:
# re-ingest the first folder — expect rows_inserted: 0
print(ingest_folder(spark, source_dir=str(sources[0]), table_name=bronze_table))

## 2. Silver — parse + topology reconstruction

Silver reads Bronze, picks the adapter from `source_format`, builds the DOM from
the raw bytes, and runs the **validated reconstruction** (vendored under
`silver/_recon/`, re-housed not re-derived). It emits four typed Delta tables and
carries the source turnover assignment as **quarantined** lineage.

We run it **in-session** (same notebook session) reading the named Bronze table —
so the Silver tables land in this session's pinned catalog and are queryable by
name.

In [ ]:
# --- run Silver Stage A+B in-session ---
from silver.notebook import reconstruct
counts = reconstruct(spark, bronze_table=bronze_table, silver_schema="silver")
print(counts)

In [ ]:
for t in ["silver_components", "silver_segments", "silver_connections", "silver_equipment"]:
    print(f"{t:22s} {spark.table('silver.' + t).count():6d} rows")

## 3. The concepts, illustrated in the data

### 3a. The crown jewel — inline valves recovered

The raw `<Connection>` records wire only each segment's two endpoints; inline valves
are missing. The reconstruction repairs the topology and re-inserts them. Here they
appear as real components flagged `is_valve` — in **both** formats.

In [ ]:
spark.table("silver.silver_components") \
     .groupBy("source_format", "is_valve").count() \
     .orderBy("source_format", "is_valve").show()

### 3b. The oracle firewall

`src_turnover` / `src_subsystem` (the source commissioning assignment) is **carried**
on the segment row — but it sits on its own columns and **nothing computes on it**.
It is the validation *answer key*, quarantined so the ~97% agreement stays honest.

In [ ]:
spark.table("silver.silver_segments") \
     .select("seg_tag", "fluid", "piping_materials_class",
             "src_turnover", "src_subsystem", "project_code").show(6, False)

### 3c. `flow_sense` — the four-state directional overlay

Direction is a *separate overlay* on the undirected connection, and it has four
states — `none` and `both` are real and a boolean couldn't hold them. Both formats
produce all four.

In [ ]:
spark.table("silver.silver_connections") \
     .groupBy("source_format", "flow_sense").count() \
     .orderBy("source_format", "flow_sense").show()

### 3d. `derived` — Source (stated) vs Derived (reconstructed) edges

Every reified connection carries provenance: `derived=false` where the edge was
stated in a source `<Connection>`, `derived=true` where the reconstruction inferred
it. This is what keeps the semantic layer from asserting inferred topology as fact.

In [ ]:
spark.table("silver.silver_connections").groupBy("source_format", "derived").count().show()

### 3e. Format parity — two standards, one schema

DEXPI and PostProc coexist in the same tables with identical columns — the
interoperability promise made concrete.

In [ ]:
spark.table("silver.silver_segments").groupBy("source_format").count().show()

### 3f. Real-data finding — `seg_tag` is not unique

Distinct `segment_id`s can compose to the **same** business `seg_tag`. So the
composed tag cannot stand alone as the CDC segment anchor — it needs a
disambiguator, and the quality gate owes an *anchor-collision* flag. (This is why
we recorded it in the spec's §3.5.)

In [ ]:
(spark.table("silver.silver_segments")
   .groupBy("seg_tag").count().filter("count > 1")
   .orderBy(F.desc("count")).show(10, False))

## 3g. Stage D — the data-quality punch list

Stage D promotes the specs' advisory flags to a **declarative expectation suite**
(rules-as-data, `silver/quality_suite.py`) and writes `silver_quality` — the
per-drawing / per-project **punch list** a pre-commissioning engineer fixes at
source *before* systemization runs (segments missing fluid / piping-class /
diameter, tags that break the naming convention, the `seg_tag` anchor-collision,
prefix-integrity, orphans). The gate is **observe-and-record**: everything flags
and flows. Only two *structural invariants* — an **oracle leak** (§5) or an
**unflagged `derived` edge** (§4) — hard-fail, because those are pipeline bugs,
not dirty data.

It also denormalises a `quality_gate` enum (`clean`/`flagged`/`quarantined`) back
onto every object row, so a cautious consumer can filter without joining the
ledger.

In [ ]:
# --- run Stage D in-session; it reads the four Silver tables ---
from silver.notebook import quality
# refdata_path lights up the reference-backed checks (unknown fluid/unit, naming);
# without it those skip cleanly. Point it at the project's Reference_Data.xlsx:
refdata_path = repo_root / "Reference_Data.xlsx"
summary = quality(spark, refdata_path=str(refdata_path) if refdata_path.exists() else None)
import json; print(json.dumps(summary, indent=2, default=str))

**The punch list** — one row per flag occurrence, ordered worst-first. This
is the artefact the engineer works from.

In [ ]:
from pyspark.sql import functions as F
sev_rank = F.when(F.col("severity") == "error", 0).when(F.col("severity") == "warn", 1).otherwise(2)
(spark.table("silver.silver_quality")
   .withColumn("_r", sev_rank)
   .orderBy("_r", "flag")
   .select("severity", "gate", "object_kind", "flag", "drawing_number", "detail")
   .show(40, False))

**Punch-list rollup by flag** — where the data gaps concentrate.

In [ ]:
(spark.table("silver.silver_quality")
   .groupBy("flag", "severity", "gate").count()
   .orderBy(F.desc("count")).show(30, False))

**The gate rollup on the objects themselves** — `flagged` rows still flow to
Gold and the rules; a strict consumer can exclude `quarantined` without a join.

In [ ]:
for t in ["silver_segments", "silver_components"]:
    print(t)
    spark.table("silver." + t).groupBy("quality_gate").count().orderBy("quality_gate").show()

## 3h. Lineage trace — one attribute, Bronze bytes → Silver column

The whole point of carrying `bronze_id` / `content_hash` on every Silver row (§4)
is that any value traces back to the exact source bytes it came from. Here we
follow **insulation** end to end: the Silver `insul_purpose` column, the raw
`InsulPurpose` attribute pulled straight out of the Bronze XML, and the `seg_tag`
suffix (e.g. `-H`) are the *same source fact reached three ways*. Reading it from
the segment's own `<GenericAttributes>` block mirrors `pidsys.master_data.ga()`
exactly. The identical three-hop walk traces fluid, diameter, piping class, or the
quarantined oracle columns — insulation isn't special.

In [ ]:
# --- Lineage trace: Insulation from Bronze bytes -> Silver columns ---
import xml.etree.ElementTree as ET

seg = spark.table("silver.silver_segments")

# 1) the Silver insulation columns + the lineage keys that trace each row to source
(seg.select("segment_id", "seg_tag", "insul_purpose", "insul_type", "insul_thick",
            "bronze_id", "content_hash", "drawing_number")
    .where("insul_purpose is not null")
    .show(8, False))

# 2) pull InsulPurpose straight out of the raw Bronze XML for one segment and compare.
bronze = spark.table(bronze_table)

row = (seg.where("insul_purpose is not null")
          .join(bronze.select("bronze_id", "content"), "bronze_id")
          .select("segment_id", "insul_purpose", "insul_type", "insul_thick", "content")
          .head())

def _ln(el):                                   # strip XML namespace
    return el.tag.split("}")[-1]

def insul_from_bytes(content, seg_id):
    # mirror pidsys.master_data.ga(): the segment's OWN <GenericAttributes> block
    root = ET.fromstring(bytes(content))
    for el in root.iter():
        if _ln(el) == "PipingNetworkSegment" and el.get("ID") == seg_id:
            return {g.get("Name"): g.get("Value")
                    for gas in el if _ln(gas) == "GenericAttributes"
                    for g in gas if _ln(g) == "GenericAttribute"
                    and (g.get("Name") or "").startswith("Insul")}
    return {}

if row is None:
    print("no segment with a non-null insul_purpose yet — run Silver Stage A+B first")
else:
    tag = seg.where(seg.segment_id == row.segment_id).head().seg_tag
    print("segment_id :", row.segment_id)
    print("seg_tag    :", tag, "  (last token = insulation purpose)")
    print("SILVER cols:", dict(insul_purpose=row.insul_purpose,
                               insul_type=row.insul_type, insul_thick=row.insul_thick))
    print("BRONZE XML :", insul_from_bytes(row.content, row.segment_id))
    # to trace a SPECIFIC flagged segment: replace the filter in `row` with
    #   .where("segment_id = '<the id from silver_quality.object_id>'")

> If `insul_purpose` comes back all-null in Silver while the `seg_tag` still
> shows a `-H`/`-N` suffix, that mismatch *is* the finding — the value reached the
> composed tag but the column extraction missed it (Bronze→Silver drift), which is
> exactly what this trace is built to catch.

## 3i. Stage C — joining the P&IDs (off-page connectors)

Stage B reconstructs each drawing on its own; **Stage C** joins them into one
plant by matching **off-page connectors** (OPCs) across sheets — by `OPCTag` for
PostProc, by GUID for DEXPI (`bppidsys.offpage.match_pairs`, re-housed). Each
matched pair becomes one undirected, always-`derived` **`OffPage`** edge in
`silver_connections` that spans two drawings; an OPC whose mate isn't in the
loaded set is an **open boundary** — the system continues off-sheet — recorded as
an `opc_open_boundary` flag, never dropped. Run it after reconstruct().

In [ ]:
# --- run Stage C in-session (harvest OPCs per sheet -> match across sheets) ---
from silver.notebook import assemble
import json
stats = assemble(spark, bronze_table=bronze_table, silver_schema="silver")
print(json.dumps(stats, indent=2, default=str))

**The cross-document edges** — one row per stitched OPC pair, `derived=true`,
joining two drawings into one connectivity graph.

In [ ]:
off = spark.table("silver.silver_connections").where("conn_type = 'OffPage'")
print("OffPage edges:", off.count())
off.select("connection_id", "from_id", "to_id", "derived", "flow_sense").show(20, False)

**Open boundaries** — OPCs with no mate in the loaded set. Not errors: the
system continues onto a sheet that wasn't loaded. Load more sheets and these
resolve into `OffPage` edges.

In [ ]:
(spark.table("silver.silver_quality").where("flag = 'opc_open_boundary'")
   .select("object_id", "drawing_number", "detail").show(20, False))

## 3j. Stage E — change data capture, a two-revision narrative

SmartPlant re-exports the **whole** drawing XML for one symbol move, and re-mints
an element's UID when it is deleted and recreated — so a file hash (or the UID)
marks everything changed. **Stage E** answers the real questions with an
*anchor-match* identity that survives delete+recreate (equipment tag / composed
seg tag / `(segment, class)` bucket) and three separated hashes: `anchor_hash`
(no UID), `content_hash_eng` (engineering attrs + neighbour **anchor** sets — the
Modify trigger), and `content_hash_audit` (adds UID + the quarantined oracle, so a
recreate is *visible* but stays *inert* for engineering CDC). It diffs, per
drawing, the two most recent Bronze versions and writes New/Modified/Deleted — the
interval open/close events Gold consumes.

To make that concrete we run a **real EPC event**: a Project-B Unit-22 (Steam &
BFW) set is **issued at Rev C for HAZOP**, then **re-issued at Rev D for design**.
Between the two, engineering changed some lines / valves / instruments / equipment
— and *every element UID is re-minted*. We ingest both revisions as two Bronze
versions and let Stage E find the real change. (Runs on its **own** `silver_cdc_demo`
schema so the main walkthrough above is untouched.)

In [ ]:
# --- generate the two-revision narrative (4 synthetic PostProc XMLs) ---
from silver.demo_cdc import write_narrative, D1, D2
paths = write_narrative(str(repo_root / "_cdc_demo"))
print("Rev C:", paths["rev1_C"]); print("Rev D:", paths["rev2_D"])

In [ ]:
# --- run the mini medallion on an ISOLATED demo schema ---
import shutil, json
from bronze.notebook import ingest_folder
from silver.notebook import reconstruct, changes

demo_bronze, demo_schema = "bronze.cdc_demo", "silver_cdc_demo"
# clean any prior demo run (tables + warehouse dirs)
spark.sql(f"DROP TABLE IF EXISTS {demo_bronze}")
shutil.rmtree(spark_warehouse / "bronze.db" / "cdc_demo", ignore_errors=True)
for t in ["silver_components","silver_segments","silver_connections","silver_equipment","silver_cdc"]:
    spark.sql(f"DROP TABLE IF EXISTS {demo_schema}.{t}")
    shutil.rmtree(spark_warehouse / f"{demo_schema}.db" / t, ignore_errors=True)

# Rev C issued -> ingest + reconstruct (the plant as HAZOP saw it)
ingest_folder(spark, source_dir=paths["rev1_C"], table_name=demo_bronze)
reconstruct(spark, bronze_table=demo_bronze, silver_schema=demo_schema)
# Rev D re-issued -> append the second version, reconstruct again
ingest_folder(spark, source_dir=paths["rev2_D"], table_name=demo_bronze)
reconstruct(spark, bronze_table=demo_bronze, silver_schema=demo_schema)

# Stage E: the change report
summary = changes(spark, bronze_table=demo_bronze, silver_schema=demo_schema)
print(json.dumps(summary, indent=2, default=str))   # expect 2 drawings, ~15 deltas

**The change report** — grouped, then in detail. Every UID changed, yet only
the real engineering changes surface (`change_type` is the Gold interval event).

In [ ]:
cdc = spark.table(f"{demo_schema}.silver_cdc")
print("total deltas:", cdc.count())
cdc.groupBy("grain", "change_type").count().orderBy("grain", "change_type").show()
(cdc.select("grain", "change_type", "drawing_number", "anchor", "detail")
    .orderBy("grain", "change_type").show(60, False))

**Churn is invisible.** Every one of the ~30 elements was re-exported with a
new UID, but the re-drawn-yet-unchanged objects (line `WBF-2215103`, pump
`P-2201A`, both lines on Drawing 0016, drum `V-2202`) produce **no delta**. The
whole of Drawing 0016 was re-issued yet only its one new PSV vent line is flagged —
no engineer has to eyeball a re-issued sheet to find what moved.

Each delta is a **Gold** interval event: *New* opens `[validFrom, ∞)`, *Deleted*
closes the prior interval, *Modified* closes the old and opens the new — giving a
defensible "current truth" *and* a Rev-C-vs-Rev-D timeline, and scoping the
change-driven work (MTO delta, the new `PSV-2201` ITR, the drum-nozzle interface,
the redlined check valve) to exactly what changed. Full manifest by discipline:
`narrative_project_b/NARRATIVE.md`.

## 4. Recap

**Built (Phase-1 + Stages C, D, E):** Bronze (raw, immutable, dedup,
format-tagged) → Silver (parse + reconstruction, four typed tables) → **Stage C
assembly** (`OffPage` edges + open boundaries) → **Stage D quality gate**
(`silver_quality` punch list + `quality_gate` rollup) → **Stage E CDC**
(`silver_cdc` object-grain deltas, delete+recreate-safe), validated on real
Project A **and** Project B. Silver is complete.

**Concepts shown:** store-as-is + content hash; format detection; the reconstruction
recovering inline valves; the oracle firewall; the `flow_sense` enum and `derived`
provenance; format parity; the `seg_tag` anchor-collision; and the Stage-D
punch list with its fail-for-bugs-not-data gate policy.

**Runtime lessons baked in:** force the venv's Spark 3.5.1 (Cell 1); pin the metastore
+ warehouse; run everything **in-session** against the named tables in that one
metastore — no path-based side door, no `!python -m …` subprocess against a live Derby.

**Next:** the **Gold layer** — bi-temporal `validFrom`/`validTo` intervals over
Stage E's deltas, then the RDF/IDO projection and the Jena rule packages
(systemization, Test Packages). Stage D's reference-backed checks light up as soon
as a project `Reference_Data.xlsx` — with `Naming` and `Insulation` sheets — is
supplied.

## 5. Gold -- bi-temporal versioning over Stage E's `silver_cdc`

> **Updated 2026-09-07 -- runs on real Spark now, not a notebook bridge.**
> Gold's bi-temporal path used to be built inline in this notebook as two
> pandas bridges (`resolve_drawing_valid_from` + `cdc_to_gold_events`,
> `.toPandas()` throughout). That's replaced below by `gold/spark_job.py`'s
> real Spark job, `run_gold(cfg, spark)` -- built to mirror
> `bronze/ingest.py` / `silver/cdc_job.py` exactly: `bronze.spark_session.get_spark`
> for the session (already running here), `.collect()` into plain dicts over
> the small per-run tables (the same discipline `silver/cdc_job.py::run_cdc`
> already uses at this PoC scale), and `spark.createDataFrame(...).write.format("delta")`
> for the write -- so Gold now follows the identical convention Bronze and
> Silver already run on in this environment, instead of pandas glue.
>
> These cells are new and unexecuted where they were written (no Spark
> session or Project A/B data available there) -- run them top-to-bottom,
> continuing from §3j above, once `gold/` (delivered separately --
> `gold_layer.zip` / `gold_layer/README.md` "Merging into the `ProjectData`
> repo") is on this repo's Python path. `gold/schema.py` and
> `gold/spark_job.py` both `import pyspark`, so they need a real Spark
> session to even import -- exactly the one this notebook already builds.

All of the resolution logic that used to live in this notebook's own Bridge
1/2 functions now lives in `gold/spark_bridge.py` instead -- a pure,
zero-pyspark-import module, unit-tested on its own
(`tests/test_spark_bridge.py`, 18 tests) without needing a `SparkSession`:

1. **valid_from** is still resolved back through **Bronze**
   (`bronze_layer_spec.md` §3.1/§6 -- Bronze is the layer that actually
   captured the revision issue date), via `spark_bridge.valid_from_by_drawing`
   -- the latest ingested revision per drawing, as the run's single
   `valid_from` (the same one-drawing-one-origin simplification
   `gold_job.py`'s own `_valid_from_for_kind()` documents for the deprecated
   snapshot-diff fallback).
2. **attrs** are resolved per grain via `spark_bridge.build_events`, exactly
   as before: `component`/`equipment`/`connection` look their attrs up by
   Stage E's `new_uid` (== that table's `{grain}_id` column); `line` grain
   resolves via `spark_bridge.resolve_line_attrs_for_event`, which prefers
   the real `silver.cdc.aggregate_lines` (imported directly when `silver/`
   is on the path) and falls back to `gold.silver_cdc.aggregate_line_attrs`
   otherwise -- the same "re-house, don't re-derive" preference this
   project settled on 2026-09-06.

`gold/temporal.py::apply_delta` is unchanged from the tested package
(`gold_layer/tests/`, 91/91 green). `run_gold` applies events through
`apply_silver_cdc_events_tolerant`, not the strict `apply_silver_cdc_events`:
a real `silver_cdc` batch's `anchor` is a **bucket key** for components
(`silver_layer_spec.md` §3.5 pairs same-class siblings on one line within
a shared bucket, not a per-instance string), so one batch can legitimately
carry two simultaneous events for one anchor -- the same non-uniqueness
§3f already documents for `seg_tag`, one layer up. The tolerant path
collects each such collision as a `CdcAnomaly` (Stage D's `silver_quality`
"flag, don't crash" precedent) instead of aborting the whole run on the
first one, and `run_gold` writes those to a `gold_anomalies` Delta table
rather than just printing them.


In [ ]:
import sys
from pathlib import Path

# gold/ ships separately (gold_layer.zip) -- point this at wherever it was
# merged into the repo. Adjust if it doesn't land at gold_layer/ next to
# bronze/ and silver/ (see gold_layer/README.md).
gold_layer_root = repo_root
if not (gold_layer_root / "gold").exists():
    raise RuntimeError(
        f"gold/ package not found at {gold_layer_root} -- merge gold_layer.zip's "
        "gold/ into the repo first (gold_layer/README.md 'Merging into the ProjectData repo')."
    )
sys.path.insert(0, str(gold_layer_root))

import gold
from gold.config import GoldConfig
from gold.spark_job import run_gold
from gold.spark_bridge import dict_to_gold_row
from gold.temporal import current_truth
print("gold package loaded from:", Path(gold.__file__).parent)
print("gold.spark_job.run_gold loaded -- the real Spark job, no pandas bridge needed")


### 5a. Running it on the Stage E Rev C -> Rev D narrative (§3j)

The demo schema already has a real `silver_cdc` from §3j's HAZOP -> design
re-issue narrative -- the natural first target. `run_gold` reads it (and
whichever per-grain Silver tables that batch touches) straight from the
catalog by name, so all this cell needs is a `GoldConfig` pointing at the
demo schema's own tables plus an isolated `gold_cdc_demo` output schema
(kept separate from `gold_schema` in §5b so the two don't collide).


In [ ]:
# same hygiene as the Bronze/Silver demo cell above (§3j) -- drop any prior
# demo Gold tables before re-running this cell.
demo_gold_schema = "gold_cdc_demo"
demo_gold_cfg = GoldConfig(bronze_table=demo_bronze, silver_schema=demo_schema,
                            gold_schema=demo_gold_schema)
for t in ("gold_objects", "gold_anomalies"):
    spark.sql(f"DROP TABLE IF EXISTS {demo_gold_cfg.table(t)}")
shutil.rmtree(spark_warehouse / f"{demo_gold_schema}.db", ignore_errors=True)

demo_summary = run_gold(demo_gold_cfg, spark=spark)
print(json.dumps(demo_summary, indent=2, default=str))

demo_gold_df = spark.table(demo_gold_cfg.table("gold_objects"))
(demo_gold_df.groupBy("object_kind", "current").count()
             .orderBy("object_kind", "current").show())

if spark.catalog.tableExists(demo_gold_cfg.table("gold_anomalies")):
    print(f"\n{demo_summary['anomalies']} anomalies (this run's Gold punch list -- not a crash):")
    spark.table(demo_gold_cfg.table("gold_anomalies")).show(50, False)


**A concrete bi-temporal query.** Pick whichever anchor actually got Modified
in this run (any anchor with more than one row-version, now read back from
the real `gold_objects` Delta table rather than kept in memory) and query
both time axes independently on it -- current truth vs. what was true during
its *first* validity window.


In [ ]:
from collections import defaultdict

rows_by_anchor = defaultdict(list)
for r in demo_gold_df.collect():
    d = r.asDict(recursive=True)
    attrs = json.loads(d["attrs_json"]) if d.get("attrs_json") else {}
    rows_by_anchor[(d["object_kind"], d["anchor_id"])].append(dict_to_gold_row(d, attrs))

demo_kind = demo_anchor = demo_history = None
for (kind, anchor), hist in rows_by_anchor.items():
    if len(hist) > 1:
        demo_kind, demo_anchor, demo_history = kind, anchor, sorted(hist, key=lambda r: r.tx_from)
        break

if demo_history is None:
    print("no anchor in this run has more than one row-version -- nothing to demonstrate a supersession with")
else:
    print(f"history for {demo_kind} {demo_anchor!r}:")
    for r in demo_history:
        print(f"  valid=[{r.valid_from}, {r.valid_to}) tx=[{r.tx_from}, {r.tx_to})  superseded_by={r.superseded_by_delta}")
    print()
    print("current truth attrs                    :", current_truth(demo_history)[0].attrs if current_truth(demo_history) else None)
    print("truth as of the FIRST row's own validity:",
          [r.attrs for r in current_truth(demo_history, as_of_valid=demo_history[0].valid_from)])


### 5b. Running it on the main Project A/B walkthrough

Stage E needs at least two ingested Bronze versions of a drawing to diff
(§3j's own framing). The main walkthrough above only ingested one version
per drawing, so this will legitimately produce nothing (or fail cleanly) the
first time -- ingest a second revision of the same drawing(s) into
`bronze_table` to exercise this for real. Left in as the natural next call,
not skipped, so the notebook already shows how the main path plugs in --
same `GoldConfig` / `run_gold` call as §5a, just pointed at the main
`bronze`/`silver` tables and a `gold` output schema instead of the demo ones.


In [ ]:
from silver.notebook import changes

main_gold_cfg = GoldConfig(bronze_table=bronze_table, silver_schema="silver", gold_schema="gold")
try:
    main_cdc_summary = changes(spark, bronze_table=bronze_table, silver_schema="silver")
    print(json.dumps(main_cdc_summary, indent=2, default=str))
    main_summary = run_gold(main_gold_cfg, spark=spark)
    print(json.dumps(main_summary, indent=2, default=str))
    if spark.catalog.tableExists(main_gold_cfg.table("gold_objects")):
        (spark.table(main_gold_cfg.table("gold_objects"))
             .groupBy("object_kind", "current").count()
             .orderBy("object_kind", "current").show())
except Exception as e:
    print(f"Stage E on the main schema needs a second Bronze revision to diff against; "
          f"skipping for now ({e}). §5a above already exercises the full path on the demo narrative.")


## 6. Gold -- the RDF/IDO projection over the current Silver snapshot

`gold/rdf_mapper.py` was written directly against `silver_layer_spec.md`
§4's table family, so it maps onto `silver_components` / `silver_segments` /
`silver_equipment` / `silver_connections` as-is (one known simplification:
`map_connection` addresses every endpoint as a component, so a `Nozzle`-type
edge's endpoint resolves to a component-shaped URI even where the true
endpoint is a nozzle -- fine for this walkthrough, worth tightening before
this feeds a real systemization run).

**Updated 2026-09-10 -- line grain now exists, on the Gold-objects-sourced
path.** The direct projection below (`build_rdf_dataset(inputs)`, reading
raw Silver tables straight into RDF with no Gold run in between) still
projects one RDF node per raw `silver_segments` *piece* -- it has no
`gold_objects` bi-temporal rows to group pieces by `(drawing_number,
seg_tag)` into a Line, so this stays a real, inherent limitation of that
path, not an oversight. The Gold-objects-sourced path
(`build_rdf_dataset_from_gold_objects`, consuming §5's `run_gold` output)
now *does* collapse pieces into one `pidsys:Line` node per anchor, with
each physical piece riding along underneath as its own real
`pidsys:PipingSegment` child node -- genuine per-piece engineering
attributes (diameter, fluid, insulation, ...), `partOf` the Line,
inheriting the Line's own bi-temporal interval (a piece has no independent
bi-temporal identity of its own -- `gold_layer_spec.md` §4.4). See §6c
below, right after the guards, for a worked example against this run's
real `gold_objects`.


In [ ]:
segments    = spark.table("silver.silver_segments").toPandas().to_dict("records")
components  = spark.table("silver.silver_components").toPandas().to_dict("records")
equipment   = spark.table("silver.silver_equipment").toPandas().to_dict("records")
connections = spark.table("silver.silver_connections").toPandas().to_dict("records")
print(f"{len(components)} components, {len(segments)} segments, "
      f"{len(equipment)} equipment, {len(connections)} connections")


In [ ]:
# graph:refdata source -- the project's own Reference_Data.xlsx (already used
# by Stage D above, §3g). Column names are printed so the renames below can
# be corrected to match this project's actual headers if they differ.
import pandas as pd

if refdata_path.exists():
    fluid_sheet = pd.read_excel(refdata_path, sheet_name="Fluid")
    boundary_sheet = pd.read_excel(refdata_path, sheet_name="Boundary")
    print("Fluid sheet columns   :", list(fluid_sheet.columns))
    print("Boundary sheet columns:", list(boundary_sheet.columns))

    # ADJUST these renames if the printed columns above differ.
    fluid_catalogue = fluid_sheet.rename(columns={
        "FluidCode": "fluid_code", "Category": "category", "Subcategory": "subcategory",
    })[["fluid_code", "category", "subcategory"]].to_dict("records")
    boundary_rows = boundary_sheet.rename(columns={
        "ComponentClass": "component_class", "Role": "role",
    })[["component_class", "role"]].to_dict("records")
else:
    fluid_catalogue, boundary_rows = [], []
    print(f"{refdata_path} not found -- graph:refdata stays empty this run "
          "(every fluid classifies as 'utility', rules_reference.py's documented fallback)")


In [ ]:
from gold.gold_job import GoldInputs, build_rdf_dataset
from gold import vocab as v

inputs = GoldInputs(
    components=components, segments=segments, equipment=equipment, connections=connections,
    fluid_catalogue=fluid_catalogue, boundary_rows=boundary_rows,
    drawing_lineage={},  # only used by the deprecated snapshot-diff path (§5 above uses silver_cdc directly)
)
ds = build_rdf_dataset(inputs)

print(f"{len(ds)} quads across {len(ds.graphs())} named graphs:")
for g in sorted(ds.graphs()):
    print(f"  {g:<45} {sum(1 for _ in ds.triples(graph=g)):>6} quads")


### 6a. The oracle-quarantine guard, on the real projection

`silver_segments.src_turnover` / `src_subsystem` are quarantined in Silver
already (§3b above); this re-asserts the same invariant at the RDF layer --
a hard failure, not a lint warning, if a mapper ever routes them outside
`graph:oracle`.


In [ ]:
from gold.oracle_guard import assert_oracle_confined
assert_oracle_confined(ds)
print("oracle-quarantine invariant holds on the real projection: "
      "src_turnover / src_subsystem never leaked outside graph:oracle")


### 6b. Declarative fluid classification & the three directional guards, on real data

Same functions validated in `gold_layer/tests/` against the synthetic
fixture, now run against this project's actual fluid catalogue and
connectivity.


In [ ]:
from collections import Counter
from gold.rules_reference import load_fluid_catalogue, classify_fluid_category, is_self_owning

catalogue = load_fluid_catalogue(ds)
seg_fluids = [s.get("fluid") for s in segments if s.get("fluid")]
dist = Counter(classify_fluid_category(f, catalogue) for f in seg_fluids)

print("segment fluid classification across the plant:")
for cat, n in dist.most_common():
    print(f"  {cat:<16} {n:5d} segments")
print("self-owning (Flare / Steam-Condensate -- never traced to a consumer):",
      sum(1 for f in seg_fluids if is_self_owning(f, catalogue)), "/", len(seg_fluids))


In [ ]:
from gold.rules_reference import flare_guard, directional_consumer_guard, relief_attribution
from gold.rdf_model import URIRef


def comp(obj_id):
    return URIRef(v.uri(v.PIDSYS + "component/", obj_id))


skipped_flare = skipped_consumer = checked = 0
for conn in connections:
    if conn.get("flow_sense") not in ("forward", "reverse", "both"):
        continue
    checked += 1
    frm, to = comp(conn["from_id"]), comp(conn["to_id"])
    if flare_guard(ds, frm, to, catalogue):
        skipped_flare += 1
    if directional_consumer_guard(ds, frm, to):
        skipped_consumer += 1

print(f"{checked} directed connections checked")
print(f"  flare_guard would skip               : {skipped_flare}")
print(f"  directional_consumer_guard would skip: {skipped_consumer}")

relief_classes = {"SafetyValveOrFitting", "Reliefdevices"}
relief_components = [c for c in components if c.get("component_class") in relief_classes]
print(f"\n{len(relief_components)} relief-class components; attribution (protected side):")
for c in relief_components[:20]:
    protected = relief_attribution(ds, comp(c["component_id"]))
    print(f"  {c['component_id']:<14} tag={str(c.get('tag')):<14} -> {protected}")


### 6c. Line/Segment containment -- the Gold-objects-sourced projection

**New 2026-09-10.** Two corrections in a row settled this design: first,
whether Silver CDC's *line*-grain rows (§5's second real-data finding, cell
recap below) should surface in RDF as `Line` **and** `Segment`, both real
nodes -- yes; second, that a `Segment` is not just an identity placeholder
under its `Line`, it carries its *own* real engineering attributes
(diameter, fluid, insulation, ...) -- confirmed directly ("The segment has
attributes").

The resulting shape: `PipingComponent --partOf--> PipingSegment --partOf-->
Line`. `gold/rdf_mapper.py::map_line` asserts one `pidsys:Line` node per
`gold_objects` line-grain anchor (its own aggregated fields -- each a tuple
of the distinct values across its pieces -- plus its own bi-temporal
interval), then calls `map_segment` once per physical piece underneath it.
Each piece is captured once, at Silver-CDC-**aggregation** time
(`gold/spark_bridge.py::resolve_line_attrs_for_event`, into the Line's own
`attrs["pieces"]`) -- never by a second, independent read of raw Silver
from the RDF-projection step, preserving the "RDF projection reads only
`gold_objects`, sequentially after `run_gold`" design this notebook's §5
already follows. A piece has no bi-temporal identity of its own: it
inherits its parent Line's exact `valid_from` / `valid_to` / `tx_from` /
`tx_to`. `build_rdf_dataset_from_gold_objects` now always projects every
line (it never skips one for lacking piece detail) and instead returns
`lines_without_piece_detail` so a caller can see which lines lack it,
rather than guessing.

See `claude/gold_layer_spec.md`'s 2026-09-11 (cont., v2) status entry for
the full design rationale and risk #10's resolution.


In [ ]:
from collections import defaultdict
from gold.gold_job import build_rdf_dataset_from_gold_objects

# reuse §5a's own demo_gold_df (gold_objects for the Rev C -> Rev D narrative) --
# this is the ONLY path with real bi-temporal rows to group pieces into a Line by;
# the direct `ds` above (§6-§6b) never gets that grouping, see §6's updated note.
gold_rows_by_kind = defaultdict(list)
for r in demo_gold_df.collect():
    d = r.asDict(recursive=True)
    attrs = json.loads(d["attrs_json"]) if d.get("attrs_json") else {}
    gold_rows_by_kind[d["object_kind"]].append(dict_to_gold_row(d, attrs))

ds_gold, lines_without_piece_detail, equipment_duplicate_components = build_rdf_dataset_from_gold_objects(
    dict(gold_rows_by_kind), fluid_catalogue, boundary_rows,
)
assert_oracle_confined(ds_gold)  # the nested per-piece Segment nodes must still route oracle fields correctly

print(f"{len(ds_gold)} quads across {len(ds_gold.graphs())} named graphs "
      "(Gold-objects-sourced, §5a's demo run):")
for g in sorted(ds_gold.graphs()):
    print(f"  {g:<45} {sum(1 for _ in ds_gold.triples(graph=g)):>6} quads")
print()
print(f"{len(lines_without_piece_detail)} line(s) projected with no piece detail "
      f"(still real RDF nodes, just flagged rather than silently guessed at): "
      f"{lines_without_piece_detail}")
print()
print(f"{len(equipment_duplicate_components)} 'component'-kind gold_objects row(s) skipped as "
      "Equipment duplicates (component_class=None, kind=='Equipment' -- the same physical "
      "item is already correctly mapped via the 'equipment' kind; see the 2026-09-10 note "
      "below): " f"{equipment_duplicate_components}")

In [ ]:
line_with_pieces = next(
    (r for r in gold_rows_by_kind.get("line", []) if r.attrs.get("pieces")), None
)

if line_with_pieces is None:
    print("no 'line' row in this run's gold_objects carries piece detail -- "
          "re-run §5a with the patched gold/spark_bridge.py: this feature attaches "
          "attrs['pieces'] at the Silver-CDC-aggregation step, it is not backfilled "
          "onto an already-written gold_objects table")
else:
    line_node = URIRef(v.uri(v.PIDSYS + "line/", line_with_pieces.anchor_id))
    print(f"Line {line_with_pieces.anchor_id!r} -- {len(line_with_pieces.attrs['pieces'])} piece(s):")
    for pred in (v.P_FLUID_CODE, v.P_UNIT, v.P_DIAMETER, v.P_PIECE_COUNT, v.P_LINE_ATTR_INCONSISTENT):
        vals = [q.o.toPython() for q in ds_gold.triples(s=line_node, p=URIRef(pred), graph=v.GRAPH_MASTERDATA)]
        if vals:
            print(f"  {pred.rsplit('#', 1)[-1]:<22} {vals}")

    print("\n  child Segment nodes -- each a real physical piece, own scalar attrs,")
    print("  partOf the Line, inheriting the Line's own bi-temporal interval:")
    for piece in line_with_pieces.attrs["pieces"]:
        seg_node = URIRef(v.uri(v.PIDSYS + "segment/", piece["segment_id"]))
        part_of = list(ds_gold.triples(s=seg_node, p=URIRef(v.P_PART_OF), o=line_node, graph=v.GRAPH_MASTERDATA))
        diameter = [q.o.toPython() for q in ds_gold.triples(s=seg_node, p=URIRef(v.P_DIAMETER), graph=v.GRAPH_MASTERDATA)]
        valid_from = [q.o.toPython() for q in ds_gold.triples(s=seg_node, p=URIRef(v.P_VALID_FROM), graph=v.GRAPH_MASTERDATA)]
        print(f"    {piece['segment_id']:<10} partOf-line={len(part_of) == 1}  "
              f"diameter={diameter}  valid_from(inherited)={valid_from}")

    # the oracle field, if this line has a piece carrying one, must land ONLY in
    # graph:oracle, never graph:masterdata -- even nested under a Line node.
    oracle_piece = next((p for p in line_with_pieces.attrs["pieces"] if p.get("src_turnover")), None)
    if oracle_piece:
        seg_node = URIRef(v.uri(v.PIDSYS + "segment/", oracle_piece["segment_id"]))
        in_masterdata = list(ds_gold.triples(s=seg_node, p=URIRef(v.P_SRC_TURNOVER_SYSTEM), graph=v.GRAPH_MASTERDATA))
        in_oracle = list(ds_gold.triples(s=seg_node, p=URIRef(v.P_SRC_TURNOVER_SYSTEM), graph=v.GRAPH_ORACLE))
        print(f"\n  oracle check on {oracle_piece['segment_id']}: "
              f"graph:masterdata={len(in_masterdata)} (must be 0), graph:oracle={len(in_oracle)} (must be 1)")


## 7. The SPARQL query surface, on the real graph

> **Updated 2026-09-09 — real SPARQL now, not just a local pattern match.**
> `Dataset` is genuinely `rdflib`-backed as of this update (see
> `gold/rdf_model.py`'s docstring for the graduation), so `sparql_queries.run_sparql`
> below runs the exact SPARQL text in `EXAMPLE_QUERIES` through rdflib's
> real query engine — no Fuseki required, and no rewriting needed when
> the same query later runs against a live Fuseki. `run_local_pattern` is
> kept alongside it for the lighter-weight lookups that don't need a full
> SPARQL string.


In [ ]:
from gold.sparql_queries import EXAMPLE_QUERIES, run_local_pattern, run_sparql

derived_hits = list(run_local_pattern(ds, p=URIRef(v.P_DERIVED), graph=v.GRAPH_MASTERDATA))
# q.o is a real rdflib Literal now -- .toPython() converts its xsd:boolean
# lexical form back to a Python bool (the old hand-rolled term's `.value`
# attribute doesn't exist on rdflib's Literal).
derived_true = sum(1 for q in derived_hits if q.o.toPython() is True)
print(f"{derived_true} / {len(derived_hits)} connections are derived "
      "(reconstruction-inferred, not source-stated)")

print()
print("the same question, via real SPARQL instead of a local pattern match:")
sparql_result = run_sparql(ds, EXAMPLE_QUERIES["derived_connections"])
print(f"{len(list(sparql_result))} derived connections returned by rdflib's SPARQL engine "
      f"(should match {derived_true} above)")

print()
print("the oracle cross-check -- the ONLY query allowed to join graph:results "
      "against graph:oracle, read-only validation reporting (needs graph:results "
      "from a real systemization run to return rows):")
print(EXAMPLE_QUERIES["oracle_cross_check"])


### 7a. OWL/RDFS entailment, via owlrl — cross-checking the class hierarchy

The first concrete step of this project's own next phase ("explore the use
of an owl Semantic approach... business rules applied to industrial plant
data based on IDO"): does a REAL OWL-RL reasoner (`owlrl`) agree that every
component/equipment/nozzle instance is transitively an `ido:PhysicalObject`,
the way `rdf_mapper.py`'s correctness note #1 and every rule in
`rules_reference.py` already assume? See `gold/owl_reasoning.py`'s module
docstring for exactly what this does and does not attempt to validate (the
fluid/flow business rules stay Jena's / `rules_reference.py`'s territory,
not OWL's).


In [ ]:
from gold.owl_reasoning import cross_check_physical_object_closure

owl_report = cross_check_physical_object_closure(ds)
print(json.dumps(owl_report, indent=2))
assert owl_report["agrees_with_project_assumption"], (
    "owlrl's real OWL-RL closure disagrees with this project's own "
    "class-hierarchy assumption -- see gold/owl_reasoning.py's docstring "
    "for how to read the report above before treating this as a false alarm."
)
print("owlrl agrees: every physical-object instance closes correctly "
      "through the asserted rdfs:subClassOf chain.")


## 8. Pushing to a real Fuseki

**Updated 2026-09-09:** this is no longer just one illustrative PUT request
-- `fuseki/docker-compose.yml` (this package's own copy, mirroring the
sibling `ido-prototype`'s `stain/jena-fuseki` setup) plus
`gold/fuseki_bootstrap.py::push_dataset` push all four named graphs at
once, drop-and-replace, safe to re-run. Still unexecuted *in this
notebook's run* -- no Fuseki instance is reachable from this sandbox
either -- but this is now real code to run against `docker compose up -d`
in `fuseki/`, not pseudocode.

**Updated 2026-09-10:** a real Fuseki instance rejects an unauthenticated
PUT with HTTP 401 -- `FusekiConfig` now takes `user`/`password` (HTTP
Basic Auth), defaulted below to match `fuseki/docker-compose.yml`'s
`ADMIN_PASSWORD`. Change both places together if you change that default.


In [ ]:
from gold.fuseki_bootstrap import push_dataset, DEFAULT_BASE_URL, DEFAULT_DATASET, DEFAULT_USER, DEFAULT_PASSWORD
from gold.fuseki_client import FusekiConfig

cfg = FusekiConfig(base_url=DEFAULT_BASE_URL, dataset=DEFAULT_DATASET,
                    user=DEFAULT_USER, password=DEFAULT_PASSWORD)  # http://localhost:3030 / "gold", admin/admin
report = push_dataset(ds, cfg)
print(json.dumps(report, indent=2))


## 9. Recap -- Gold added

**Updated 2026-09-07 -- §5 now calls the real Spark job, not the old pandas
bridges.** An earlier run of §5 (when it still built `resolve_drawing_valid_from`
/ `cdc_to_gold_events` inline via `.toPandas()`) was executed once against
the real Rev C -> Rev D narrative and produced 15 Gold events / 1 anchor
collision on a batch where every anchor was a first-time `New` -- correctly
showing no supersession, since that batch had nothing to supersede yet (see
`bronze_silver_gold_rev_c_to_d.ipynb` for a small fixture built specifically
to show a genuine `Modified` instead). §5's cells are rewritten as of today
to call `gold/spark_job.py::run_gold` / `gold/config.py::GoldConfig`
directly -- unexecuted again where they were rewritten (same reason as
before: no Spark session here), so re-run §5 top-to-bottom once more before
relying on its output; §6-§8 remain unexecuted for the same reason and are
otherwise unchanged by today's edit.

**Built above (new):** bi-temporal versioning consuming Stage E's real
`silver_cdc` directly, now via a real Spark job (§5 -- `run_gold`, backed by
`gold/spark_bridge.py`'s resolution logic rather than bridges built inline
in the notebook), the RDF/IDO projection over the real Silver snapshot
(§6), the oracle-quarantine guard re-asserted at the RDF layer and passing
(§6a), the fluid classification and three directional guards run against
real connectivity (§6b), the SPARQL query surface (§7), and an illustrative
Fuseki push (§8).

**A real-data finding from that first run, now handled, not worked around:**
`silver_cdc`'s `anchor` is a bucket key for components -- two same-class
siblings on one line can share it (`silver_layer_spec.md` §3.5, the same
non-uniqueness §3f already documents for `seg_tag`), so a strict
`apply_delta` call can hit a real "NEW delta for anchor already current"
collision. §5 now applies events through `apply_silver_cdc_events_tolerant`
(`gold/silver_cdc.py`), which flags each such collision as a `CdcAnomaly`
instead of aborting the batch -- this project's own Stage-D "observe and
record, don't crash on dirty data" discipline.

**A second real-data finding, from Silver's 2026-09-05 line-grain rename:**
Silver's piping CDC moved from per-physical-segment grain to **line** grain
(`(drawing_number, seg_tag)`, collapsing the ~78% of seg_tags shared by 2+
physical pieces). A fresh `grain='line'` sample (2026-09-06) showed
`old_uid`/`new_uid` at this grain is `f"{drawing_number}|{seg_tag}"`, not a
`silver_segments` row id -- the old single-row `.loc[new_uid]` attrs lookup
in `cdc_to_gold_events` (§5's Bridge 2) could never have worked for it.
Fixed: `gold/silver_cdc.py::OBJECT_KINDS` now accepts `"line"` (not
`"segment"`), and `aggregate_line_attrs` groups the matching
`silver_segments` pieces by `(drawing_number, seg_tag)` and reduces each
engineering attribute to a distinct-value tuple, mirroring Silver's own
`content_hash_eng` set-projection (`line_attr_inconsistent` surfaces a
within-line disagreement rather than picking a value). One thing this
*didn't* need to change: `anchor` is opaque to Gold everywhere, so whether
a `grain='component'` anchor is `CMP|SEG|...` or `CMP|LINE|...`
post-rename makes no functional difference here.

69/69 tests green at the time of that finding (9 new: 3 for the
anchor-bucket collision, 6 for the line-grain resolution helpers --
`tests/test_silver_cdc.py`); **91/91 as of today** (2026-09-07), after
`silver.cdc.aggregate_lines` was confirmed importable (below) and
`gold/spark_bridge.py`'s 18 tests were added for the Spark-job rewrite.

**Resolved 2026-09-06 (was "still open" here):** `silver.cdc.aggregate_lines`
is real, pure Python, dependency-free, and importable as-is -- confirmed
once the `ProjectData` GitHub sync widened to include `/silver/`. §5's
resolution logic (now `gold/spark_bridge.py::lines_for_drawing` /
`resolve_line_attrs_for_event`, at the time still `cdc_to_gold_events`
inline in this notebook) calls it directly whenever importable, falling
back to `gold.silver_cdc.aggregate_line_attrs`'s re-derivation only for a
`gold_layer`-only checkout without `silver/` alongside it.

**Known gaps, left honest rather than papered over:** `map_connection`
addresses every endpoint as a component (Nozzle-typed edges need a real
`from_kind`/`to_kind` before this feeds a systemization run); the
`Reference_Data.xlsx` column renames in §6 are guesses -- check the printed
`.columns` output; §5b's main-walkthrough Stage E call needs a second
ingested Bronze revision to do anything; `graph:results` (computed
commissioning systems) isn't populated by anything in this notebook yet --
that's the walk.py global fragment-partition step the strategy keeps in
Python (`medallion_rdf_ido_strategy_mapping.md` §6), not shown here.

**Updated 2026-09-09 -- §7a (OWL-RL cross-check via `owlrl`) and a real
Fuseki setup are new.** `gold/rdf_model.py` graduated off its hand-rolled
quad store onto real `rdflib`, at the user's request, now that their own
local environment carries `rdflib`/`owlrl`/`pyspark`/`delta-spark` (this
sandbox still doesn't -- every cell above stays unexecuted here for that
reason). §8's Fuseki push is no longer one illustrative PUT: `fuseki/docker-
compose.yml` (this package's own copy, mirroring the sibling `ido-prototype`'s
`stain/jena-fuseki` setup one-for-one) plus `gold/fuseki_bootstrap.py::
push_dataset` push all four named graphs at once, drop-and-replace, safe to
re-run -- `python -m gold.fuseki_bootstrap --fixture` is a standalone smoke
test against the same `tests/fixtures.py` scenario this notebook itself
builds `ds` from. This directly targets the "live Fuseki/Jena round-trip"
gap named below and in `gold_layer_spec.md` §8.2/§10 -- run it against your
own `docker compose up -d` to close it for real, something this sandbox
cannot do.

**Updated 2026-09-10 -- §6c is new: Line/Segment containment, illustrated
against real `gold_objects`.** Two corrections in a row drove this:
whether Silver CDC's line-grain rows should surface in RDF as `Line` *and*
`Segment`, both real nodes (yes), and then that a `Segment` carries its own
real engineering attributes rather than being an identity-only placeholder
under its `Line` ("The segment has attributes"). `gold/rdf_mapper.py::map_line`
(new) asserts one `pidsys:Line` node per `gold_objects` line-grain anchor
and calls the extended `map_segment` once per physical piece underneath it
-- each piece a real `pidsys:PipingSegment` child node, `partOf` the Line,
carrying its own scalar attributes (six new predicates: `unit`, `diameter`,
`pipingMaterialsClass`, `insulationType`/`insulationPurpose`/
`insulationThickness`) and inheriting the Line's bi-temporal interval
rather than tracking one of its own. The pieces are captured once, at
Silver-CDC-aggregation time (`gold/spark_bridge.py::resolve_line_attrs_for_event`,
into the Line's own `attrs["pieces"]`) -- never by a second read of raw
Silver from the RDF-projection step, preserving §5's "RDF projection reads
only `gold_objects`" design. `build_rdf_dataset_from_gold_objects` now
always projects every line and reports `lines_without_piece_detail`
instead of silently skipping one. §6's opening note above is corrected in
place rather than left to mislead a later reader. 6 new tests
(`tests/test_gold_job.py`, `tests/test_spark_bridge.py`); see
`claude/gold_layer_spec.md`'s 2026-09-11 (cont., v2) entry for the full
design rationale and risk #10's resolution.

**Updated 2026-09-10 (cont.) -- `component_class=None` root-caused and
handled, not papered over.** Running §6c above against real
`silver_components` crashed with `KeyError: 'component_class'` --
`gold/spark_bridge.py::build_events` drops any `None`-valued Silver column
from `attrs` entirely (key absent, not `None`), and `map_component` read
`comp["component_class"]` unconditionally. Two distinct issues, both fixed:
(1) `component_class` is genuinely absent on several element kinds by
design -- `silver/_recon/{pidtool,bppidsys}/model.py::COMPONENT_TAGS`
extracts Nozzle/Equipment/PropertyBreak/off-page-connector/instrumentation
nodes purely for connectivity, but `ComponentClass` only exists on
`PipingComponent`/`ProcessInstrument`/`InstrumentComponent` per the
DEXPI/PostProc schema, so `Doc.cc(e)` returns `None` for the rest --
`map_component` now treats `component_class` as optional and falls back to
a new, honest `pidsys:UnclassifiedComponent` RDF type (subclass of
`pidsys:PipingComponent`) rather than crashing or guessing. (2) Checked
against the actual reconstruction pipeline (Rev C/D narrative fixtures):
**100% of the null-class rows are `kind=="Equipment"`**, and each is a
genuine duplicate of the corresponding `silver_equipment` row -- the same
physical item, reconstructed twice because `silver/reconstruct.py`'s
component-building loop only special-cases `kind=="Segment"`, so
Equipment-kind elements land in both `silver_components` (classless) and
`silver_equipment` (correctly typed, via `pidsys/reconstructed.py::
_wire_equipment`'s ghost-filtered path). Decision (confirmed): exclude
`kind=="Equipment"` rows from the component projection entirely rather
than double-map the same item under two RDF types --
`build_rdf_dataset_from_gold_objects` now returns a third element,
`equipment_duplicate_components`, reporting every anchor id skipped this
way (same "observe and record" discipline as `lines_without_piece_detail`),
and §6c's cell above prints it. See `claude/gold_layer_spec.md`'s matching
2026-09-10 entry for the full write-up.

See `claude/gold_layer_spec.md` for the full design rationale, open risks
(§10), and `gold_layer/medallion_concepts.ipynb` (delivered alongside this
notebook) for the same Gold functionality exercised standalone against a
small synthetic fixture, with every cell actually executed.


In [ ]:
"""One-off diagnostic: real distribution of component_class=None rows by `kind`,
for Project B (PostProc) drawings, against real silver_components / gold_objects.
Run inside your Spark session (or adapt the read to however you load these tables).
"""
from pyspark.sql import functions as F

# Option A: against silver_components directly (source_format filter optional)
df = spark.table("silver.silver_components")
# df = df.where(F.col("source_format") == "POSTPROC")  # uncomment to scope to Project B only

dist = (
    df.groupBy("kind")
      .agg(
          F.count("*").alias("total_rows"),
          F.sum(F.when(F.col("component_class").isNull(), 1).otherwise(0)).alias("null_class_rows"),
      )
      .withColumn("pct_null", F.round(100 * F.col("null_class_rows") / F.col("total_rows"), 1))
      .orderBy(F.desc("null_class_rows"))
)
dist.show(20, False)

spark.table("silver.silver_components").where("kind='Equipment'").select("component_class").distinct().show(20, False)

In [ ]:
"""Broader diagnostic: find where genuinely-unclassified component rows
actually live, across ALL documents, by kind AND by drawing -- and catch
"catch-all" string values (Custom/Generic/etc.), not just SQL NULL.

Run inside your Spark session.
"""
from pyspark.sql import functions as F

df = spark.table("silver.silver_components")

# --- Part 1: which VALUES actually show up per kind? -----------------------
# A strict `component_class IS NULL` check (as run before) can miss a
# catch-all string the source export uses instead of a true null --
# ido_semantic_mapping_spec.md's own real-data finding (§4.4) calls this
# "generic/Custom/None class" as one bucket, not three separate ones.
CATCHALL_LIKE = (
    F.col("component_class").isNull()
    | (F.trim(F.col("component_class")) == "")
    | F.lower(F.col("component_class")).isin("custom", "generic", "none", "unknown", "n/a", "na")
)

print("=== distinct component_class values per kind (look for catch-alls) ===")
(df.groupBy("kind", "component_class")
   .count()
   .orderBy("kind", F.desc("count"))
   .show(200, False))

print("=== catch-all-like rows by kind (broader than a plain null check) ===")
(df.groupBy("kind")
   .agg(
       F.count("*").alias("total_rows"),
       F.sum(F.when(CATCHALL_LIKE, 1).otherwise(0)).alias("catchall_rows"),
   )
   .withColumn("pct_catchall", F.round(100 * F.col("catchall_rows") / F.col("total_rows"), 1))
   .orderBy(F.desc("catchall_rows"))
   .show(20, False))

# --- Part 2: which DRAWINGS actually have catch-all rows? -------------------
# Use this to pick a real drawing to inspect closely, rather than whichever
# table/batch happened to be queried last.
drawing_col = "drawing_number" if "drawing_number" in df.columns else "document_number"
print(f"=== drawings with the most catch-all-like component rows (by {drawing_col}) ===")
(df.where(CATCHALL_LIKE)
   .groupBy(drawing_col, "kind")
   .count()
   .orderBy(F.desc("count"))
   .show(40, False))

# --- Part 3: the specific drawing ido_semantic_mapping_spec.md already flagged ---
# Real, documented finding (§4.4): 362-09-01010 (Project A / DEXPI) has 132/478
# items (28%) in the "generic/Custom/None class" bucket. Check its kind
# distribution directly -- this is the best-known real candidate for a
# genuinely unclassified, non-Equipment item.
print("=== drawing 362-09-01010 specifically (ido_semantic_mapping_spec.md §4.4) ===")
target = df.where(F.col(drawing_col).contains("362-09-01010"))
if target.limit(1).count() == 0:
    print("(not found under this exact drawing_number/document_number value -- "
          "check the actual column value, e.g. via df.select(drawing_col).distinct().show(50, False))")
else:
    (target.groupBy("kind", "component_class")
           .count()
           .orderBy("kind", F.desc("count"))
           .show(100, False))

# --- Part 4: Project A vs Project B side by side, if source_format exists ---
if "source_format" in df.columns:
    print("=== catch-all rate by kind AND source_format (DEXPI vs PostProc) ===")
    (df.groupBy("source_format", "kind")
       .agg(
           F.count("*").alias("total_rows"),
           F.sum(F.when(CATCHALL_LIKE, 1).otherwise(0)).alias("catchall_rows"),
       )
       .withColumn("pct_catchall", F.round(100 * F.col("catchall_rows") / F.col("total_rows"), 1))
       .orderBy("source_format", F.desc("catchall_rows"))
       .show(40, False))

In [ ]:
# Option B: same question against gold_objects (component kind only), if you'd rather
# check post-Gold-projection state (attrs is a map/struct column there):
# god = spark.table("gold.gold_objects").where("object_kind = 'component'")
# god.groupBy(F.col("attrs.kind")).agg(
#     F.count("*").alias("total_rows"),
#     F.sum(F.when(F.col("attrs.component_class").isNull(), 1).otherwise(0)).alias("null_class_rows"),
# ).show(20, False)
